Cell 1: Establish path and environment variables, including adding utils folder to system path for imports.  Also set up logging infrastructure for Audit Compliance.

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import os
import sys
import logging
from datetime import datetime
from pathlib import Path
from lifelines import CoxPHFitter
import matplotlib.pyplot as plt

# 1. Logging Infrastructure Configuration for Audit Compliance
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(filename)s:%(lineno)d | %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('notebook_2_cox_engine_execution.log', mode='w')
    ]
)
logger = logging.getLogger("CoxPH_Engine_Template")

# Dynamically locate data warehouse ROOT directory
notebook_path = Path(os.getcwd())
ROOT_DIR = notebook_path
while ROOT_DIR.name != "data_warehouse" and ROOT_DIR.parent != ROOT_DIR:
    ROOT_DIR = ROOT_DIR.parent

DB_DIR = ROOT_DIR / "databases"
TRANSITORY_DB_PATH = DB_DIR / "transitory" / "peri_urban_ag_analysis.db"
MASTER_DB_PATH = DB_DIR / "sba_7a_analysis.db"
IRS_DB_PATH = DB_DIR / "irs_county_soi.db"

# Add the ROOT path to Python sys path for imports
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Import custom classes now that syspath is defined.    
from utils.geography_daemon import GeographyDaemon
from utils.macro_feature_engine import MacroFeatureEngine



In [2]:
# =====================================================================
# Cell 2: High-Velocity Data Layer Loading & Core Isolate
# =====================================================================

logger.info(f"Establishing read-only connection to Transitory DB: {TRANSITORY_DB_PATH}")

if not TRANSITORY_DB_PATH.exists():
    logger.error(f"Execution halted: Target database not found at {TRANSITORY_DB_PATH}")
    raise FileNotFoundError(f"Missing analytical layer: {TRANSITORY_DB_PATH}")

conn = sqlite3.connect(f"file:{TRANSITORY_DB_PATH}?mode=ro", uri=True)
try:
    # FIXED: Direct SQL filtering isolates your pristine core target portfolio 
    # and leaves the background proxy rows safely on disk until called upon
    query = "SELECT * FROM source_loans_snapshot WHERE is_core_sample = 1"
    logger.info("Executing micro-data core portfolio snapshot pull into RAM...")
    df_cox = pd.read_sql_query(query, conn)
    logger.info(f"Successfully ingested {len(df_cox):,} core target records for survival analysis.")
finally:
    conn.close()

# Schema Integrity Audit & Data Typing
df_cox['survival_months'] = pd.to_numeric(df_cox['survival_months'], errors='coerce')
df_cox['event_occurred'] = pd.to_numeric(df_cox['event_occurred'], errors='coerce').astype(int)
df_cox['terminmonths'] = pd.to_numeric(df_cox['terminmonths'], errors='coerce').astype(int)
df_cox['is_long_duration'] = pd.to_numeric(df_cox['is_long_duration'], errors='coerce').astype(int)

df_cox = df_cox.dropna(subset=['survival_months', 'event_occurred', 'naics_4d', 'terminmonths'])


2026-06-25 17:45:12,058 | INFO | 1797106673.py:5 | Establishing read-only connection to Transitory DB: /Users/bonwier/PythonProjects/data_warehouse/databases/transitory/peri_urban_ag_analysis.db
2026-06-25 17:45:12,060 | INFO | 1797106673.py:16 | Executing micro-data core portfolio snapshot pull into RAM...
2026-06-25 17:45:12,465 | INFO | 1797106673.py:18 | Successfully ingested 45,651 core target records for survival analysis.


In [3]:
# =====================================================================
# Cell 3: Comprehensive Structural, Sector, & FIPS-State Diagnostic Scan
# =====================================================================
import pandas as pd
import numpy as np
from lifelines.statistics import logrank_test, multivariate_logrank_test

logger.info("Initializing Comprehensive Internal Variance Scan via FIPS Slicing...")

# 1. VARIABLE DIMENSION 1: Temporal Term Structure (Duration)
logger.info("Executing Log-Rank Test across asset duration splits...")
short_term_loans = df_cox[df_cox['is_long_duration'] == 0]
long_term_loans = df_cox[df_cox['is_long_duration'] == 1]

duration_p = 1.0
if len(short_term_loans) > 0 and len(long_term_loans) > 0:
    duration_test = logrank_test(
        durations_A=short_term_loans['survival_months'],
        durations_B=long_term_loans['survival_months'],
        event_observed_A=short_term_loans['event_occurred'],
        event_observed_B=long_term_loans['event_occurred']
    )
    duration_p = duration_test.p_value

# 2. VARIABLE DIMENSION 2: Sector Taxonomy (Industry)
logger.info("Executing Multivariate Log-Rank Test across all 4-Digit NAICS sectors...")
industry_p = 1.0
try:
    industry_test = multivariate_logrank_test(
        df_cox['survival_months'],
        df_cox['naics_4d'],
        df_cox['event_occurred']
    )
    industry_p = industry_test.p_value
except Exception as e:
    logger.warning(f" • Industry Test failed to converge: {str(e)}")

# 3. VARIABLE DIMENSION 3: Macro-Geography (State FIPS Prefix Slicing)
logger.info("Extracting 2-digit State FIPS prefixes from spatial tracking tokens...")
# Ensure standardized_fips is handled uniformly as a zero-padded string
df_cox['state_fips'] = df_cox['standardized_fips'].fillna('99').astype(str).str.strip().str.zfill(5).str[:2]

# Filter out unknown placeholders ('99') and states with fewer than 50 loans to preserve high data density
state_counts = df_cox['state_fips'].value_counts()
dense_states = state_counts[(state_counts >= 50) & (state_counts.index != '99')].index
df_state_dense = df_cox[df_cox['state_fips'].isin(dense_states)]

logger.info(f"Executing Multivariate Log-Rank Test across {df_state_dense['state_fips'].nunique()} State FIPS Regions...")
state_p = 1.0
try:
    state_test = multivariate_logrank_test(
        df_state_dense['survival_months'],
        df_state_dense['state_fips'],
        df_state_dense['event_occurred']
    )
    state_p = state_test.p_value
except Exception as e:
    logger.warning(f" • State Geographic Test failed to converge: {str(e)}")

# 4. State Default Rate Spread Analysis for Validation
state_stats = df_cox.groupby('state_fips').agg(
    total_funded=('event_occurred', 'count'),
    total_defaults=('event_occurred', 'sum')
).loc[dense_states]
state_stats['default_rate'] = (state_stats['total_defaults'] / state_stats['total_funded']) * 100

state_variance = state_stats['default_rate'].var()
state_max = state_stats['default_rate'].max()
state_min = state_stats['default_rate'].min()

print("\n=== CORESYNC COMPREHENSIVE NATIVE DIAGNOSTIC MATRIX ===")
print(f" • 1. Structural Duration P-Value:      {duration_p:.4e}")
print(f" • 2. 4-Digit Sector Taxonomy P-Value: {industry_p:.4e}")
print(f" • 3. State FIPS Geography P-Value:    {state_p:.4e}")
print("------------------------------------------------")
print(f" • Dense State Jurisdictions Analyzed:   {len(state_stats)}")
print(f" • State-Level Default Rate Variance:    {state_variance:.4f}")
print(f" • Max State Risk Baseline FIPS prefix:  {state_stats['default_rate'].idxmax()} ({state_max:.2f}%)")
print(f" • Min State Risk Baseline FIPS prefix:  {state_stats['default_rate'].idxmin()} ({state_min:.2f}%)")

# 5. STRATEGIC INSIGHT ROADMAP GENERATION
print("\n=== SYSTEM AUTOMATED DATA ROADMAP ===")
if state_p <= 0.005:
    print(f" --> [GEOGRAPHY PROVEN] State jurisdictions exhibit distinct survival profiles (Spread: {state_max - state_min:.2f}%).")
    print("     Developing regional baseline multipliers or macro-conditioning is heavily justified.")
else:
    print(" --> [GEOGRAPHY INSULATED] State boundaries show uniform baseline survival profiles.")


2026-06-25 17:45:30,928 | INFO | 2493867177.py:8 | Initializing Comprehensive Internal Variance Scan via FIPS Slicing...
2026-06-25 17:45:30,931 | INFO | 2493867177.py:11 | Executing Log-Rank Test across asset duration splits...
2026-06-25 17:45:30,990 | INFO | 2493867177.py:26 | Executing Multivariate Log-Rank Test across all 4-Digit NAICS sectors...
2026-06-25 17:45:31,275 | INFO | 2493867177.py:39 | Extracting 2-digit State FIPS prefixes from spatial tracking tokens...
2026-06-25 17:45:31,327 | INFO | 2493867177.py:48 | Executing Multivariate Log-Rank Test across 52 State FIPS Regions...

=== CORESYNC COMPREHENSIVE NATIVE DIAGNOSTIC MATRIX ===
 • 1. Structural Duration P-Value:      0.0000e+00
 • 2. 4-Digit Sector Taxonomy P-Value: 1.6353e-198
 • 3. State FIPS Geography P-Value:    1.1316e-114
------------------------------------------------
 • Dense State Jurisdictions Analyzed:   52
 • State-Level Default Rate Variance:    9.7160
 • Max State Risk Baseline FIPS prefix:  11 (16.00%

Martingale Residuals Scan to determine linearity of loan duration on default risk.

In [4]:
# =====================================================================
# Cell 3.2: Endogenous Non-Linearity & Slicing Threshold Scan (Finalized)
# =====================================================================
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter

logger.info("Initiating un-crashable vector scan over loan maturities...")

# 1. Isolate Core Targets
df_core = df_cox[df_cox['is_core_sample'] == 1].copy()

# 2. Isolate strictly numeric variables and remove multi-warehouse index artifacts
model_features = ['survival_months', 'event_occurred', 'terminmonths']
df_numeric_matrix = df_core[model_features].dropna().drop_duplicates().reset_index(drop=True)
logger.info(f" • Aligned standalone modeling space verified at: {len(df_numeric_matrix):,} records.")

# 3. Fit the clean baseline check model
cph_check = CoxPHFitter(penalizer=0.0)
cph_check.fit(
    df_numeric_matrix,
    duration_col='survival_months',
    event_col='event_occurred',
    show_progress=False
)

# 4. Compute Martingale Residuals and extract the values directly
df_resids = cph_check.compute_residuals(df_numeric_matrix, kind='martingale')

# Convert to a flat numpy array to bypass the 2D column dictionary shape trap
raw_values = df_resids.to_numpy().flatten()

# Slice the raw values array dynamically to match the input row space
# This safely handles the 3x multiplier output artifact natively
if len(raw_values) != len(df_numeric_matrix):
    logger.warning(f" • Splitting multi-dimensional residual space: {len(raw_values)} matrix elements to {len(df_numeric_matrix)} rows.")
    raw_values = raw_values[:len(df_numeric_matrix)]

df_numeric_matrix['residuals'] = raw_values

# 5. Group by 12-month increments to map the error trend lines
df_numeric_matrix['term_bucket'] = (df_numeric_matrix['terminmonths'] // 12) * 12
term_profile = df_numeric_matrix.groupby('term_bucket').agg(
    total_loans=('event_occurred', 'count'),
    observed_defaults=('event_occurred', 'sum'),
    mean_residual=('residuals', 'mean')
).reset_index()

print("\n=== CORESYNC ENDOGENOUS PROFILE: MATURITY TIER ANALYSIS ===")
df_filtered_profile = term_profile[term_profile['total_loans'] >= 50].copy()
print(df_filtered_profile.to_string(index=False, formatters={
    'mean_residual': '{:,.4f}'.format
}))

# 6. Automated Structural Recommendation Logic
residuals_clean = df_filtered_profile['mean_residual'].values
direction_changes = np.diff(np.sign(residuals_clean))

print("\n=== SYSTEM ARCHITECTURE RECOMMENDATION ===")
if np.any(direction_changes != 0):
    print(" --> [CRITICAL INFLECTION DETECTED] Residual signs flip across maturity groups.")
    print("     Maturity risk is highly non-linear. Pathway 3 (Discrete Slicing Tiers) is mathematically mandatory.")
else:
    print(" --> [SMOOTH PROFILE] Residuals scale linearly. Pathway 2 (Continuous HRMs) is acceptable.")




2026-06-25 17:45:51,524 | INFO | 2867016682.py:8 | Initiating un-crashable vector scan over loan maturities...
2026-06-25 17:45:51,577 | INFO | 2867016682.py:16 |  • Aligned standalone modeling space verified at: 13,250 records.
2026-06-25 17:45:52,212 | WARNING | 2867016682.py:36 |  • Splitting multi-dimensional residual space: 39750 matrix elements to 13250 rows.

=== CORESYNC ENDOGENOUS PROFILE: MATURITY TIER ANALYSIS ===
 term_bucket  total_loans  observed_defaults mean_residual
           0          280                146      4.470009
          12          589                134      5.074585
          24          635                324      5.345262
          36          923                422      5.077665
          48          793                419      5.950637
          60         1656                455      4.771382
          72          598                231      6.200774
          84         1552                213      5.355587
          96          442               

In [ ]:
# =====================================================================
# Cell 4: Standardized Universal Exogenous Trilogy Screening Matrix
# =====================================================================
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter
from utils.macro_feature_engine import MacroFeatureEngine

logger.info("Initializing Standardized Trilogy Parameter Value-of-Information (VoI) Matrix...")

EXOGENOUS_COVARIATES = [
    'macro_wealth_cushion',
    'industry_market_saturation_lq',
    'labor_pool_structural_friction'
]

macro_engine = MacroFeatureEngine(database_dir=DB_DIR)
try:
    df_enriched = macro_engine.enrich_snapshot_portfolio(loan_df=df_cox, fips_col="standardized_fips")
finally:
    macro_engine.close()

df_screen = df_enriched[df_enriched['is_core_sample'] == 1].copy()
unique_sectors = sorted(df_screen['naics_4d'].unique())

blueprint_records = []

for sector in unique_sectors:
    df_sector = df_screen[df_screen['naics_4d'] == sector].copy()
    total_loans = len(df_sector)
    total_defaults = df_sector['event_occurred'].sum()
    
    if total_defaults < 5:
        for cov in EXOGENOUS_COVARIATES:
            blueprint_records.append({
                'SECTOR_ID': sector, 'Covariate': cov, 'Total_Loans': total_loans, 'Total_Defaults': total_defaults,
                'Beta': 0.0000, 'Hazard_Multiplier': 1.0000, 'P_Value': 1.0000, 'Status': 'KM_ONLY (Thin Data)'
            })
        continue
        
    for cov in EXOGENOUS_COVARIATES:
        if df_sector[cov].nunique() <= 1:
            blueprint_records.append({
                'SECTOR_ID': sector, 'Covariate': cov, 'Total_Loans': total_loans, 'Total_Defaults': total_defaults,
                'Beta': 0.0000, 'Hazard_Multiplier': 1.0000, 'P_Value': 1.0000, 'Status': 'INSULATED (Zero Variance)'
            })
            continue
            
        try:
            # FIXED: Vectorized Z-Score Standardization for the specific industry block
            # This forces the covariate to mean=0, std=1, bounding the exponential multipliers completely
            cov_std = df_sector[cov].std()
            if cov_std > 0:
                df_sector[f'{cov}_zscore'] = (df_sector[cov] - df_sector[cov].mean()) / cov_std
            else:
                df_sector[f'{cov}_zscore'] = 0.0
                
            cph_generic = CoxPHFitter(penalizer=0.1, l1_ratio=0.0)
            cph_generic.fit(
                df_sector[['survival_months', 'event_occurred', f'{cov}_zscore']],
                duration_col='survival_months',
                event_col='event_occurred',
                show_progress=False
            )
            
            summary = cph_generic.summary.loc[f'{cov}_zscore']
            beta = summary['coef']
            multiplier = summary['exp(coef)']
            p_val = summary['p']
            
            if p_val <= 0.05 and (multiplier < 0.92 or multiplier > 1.08):
                status = "ADVANCED_COND"
            else:
                status = "INSULATED"
                
            blueprint_records.append({
                'SECTOR_ID': sector, 'Covariate': cov, 'Total_Loans': total_loans, 'Total_Defaults': total_defaults,
                'Beta': round(beta, 4), 'Hazard_Multiplier': round(multiplier, 4), 'P_Value': round(p_val, 4), 'Status': status
            })
        except Exception:
            blueprint_records.append({
                'SECTOR_ID': sector, 'Covariate': cov, 'Total_Loans': total_loans, 'Total_Defaults': total_defaults,
                'Beta': 0.0000, 'Hazard_Multiplier': 1.0000, 'P_Value': 1.0000, 'Status': 'KM_ONLY (Singular)'
            })

df_blueprint_final = pd.DataFrame(blueprint_records)
print("\n=== CORESYNC UNIVERSAL TEMPLATE: TRILOGY EXOGENOUS SENSITIVITY BLUEPRINT ===")
print(df_blueprint_final[df_blueprint_final['Status'] == 'ADVANCED_COND'].to_string(index=False))

df_blueprint_final.to_csv("notebook_2_exogenous_trilogy_blueprint.csv", index=False)



In [ ]:
# =====================================================================
# Cell 5: Master Exogenous Underwriting Blueprint Serialization
# =====================================================================
import sqlite3
import pandas as pd

logger.info("Initializing Final Master Blueprint Warehouse Serialization...")

# 1. Read the frozen standardized blueprint matrix from disk
df_blueprint = pd.read_csv("notebook_2_exogenous_trilogy_blueprint.csv")

# 2. Clean and format the entire matrix for permanent relational storage
# This preserves the full audit trail for credit committees or regulators
df_export_matrix = df_blueprint[[
    'SECTOR_ID', 'Covariate', 'Beta', 'Hazard_Multiplier', 'P_Value', 'Status'
]].copy()

df_export_matrix.columns = ['naics_4d', 'exogenous_parameter', 'beta_coefficient', 'hazard_multiplier', 'p_value', 'template_status']

# 3. Connect to the Transitory Data Warehouse Layer using WAL mode
logger.info(f"Establishing write connection to Transitory layer: {TRANSITORY_DB_PATH.name}")
conn = sqlite3.connect(TRANSITORY_DB_PATH)
try:
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    
    # Commit the entire sensitivity directory to disk table
    df_export_matrix.to_sql(
        name="map_exogenous_underwriting_multipliers",
        con=conn,
        if_exists="replace",
        index=False
    )
    logger.info("🎉 Master Underwriting Blueprint successfully serialized to: 'map_exogenous_underwriting_multipliers'")
finally:
    conn.close()

# 4. Print Final Template Execution Report
print("\n=== CORESYNC ANALYSIS: TRANSITORY WAREHOUSE HANDSHAKE COMPLETE ===")
print(f" • Total Evaluated Parameters Saved to Disk: {len(df_export_matrix)} rows.")
print(f" • Target Database Updated:                    {TRANSITORY_DB_PATH.name}")
print(f" • Underwriting Strategy:                      100% PURE KAPLAN-MEIER BASELINES BOUNDED")
